# Settling-column bathymetry analysis

Plots CHIRP sonar depth readings from the settling-column tests described in
[`README.md`](README.md):

1. **Average depth vs. sediment height**: mean depth of each scan with error bars of **±1 sample standard deviation** (`ddof=1`), against the depth expected from the measured water and sediment heights.
2. **Depth over time**: every scan overlaid on a common elapsed-time axis.
3. **Manual plot**: any list of `bathymetry.csv` files, ignoring the folder naming convention.

Loading and statistics come from [`chirp_bathymetry.py`](chirp_bathymetry.py). It applies the same documented rules as the web and mobile apps (see the root README, section *Data processing*): columns read by header name or validated position, samples sorted by time, small time islands trimmed, and depth measured downward from the transducer face. Install the requirements with `pip install -r requirements.txt`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, MultipleLocator

import chirp_bathymetry as cb

SCANS_DIR = Path.cwd() / "scans"

# Water height above the column floor at the start of each test, in cm
# (the sonar rests at the water surface). Replace with your value.
OBSERVED_WATER_HEIGHT_CM = 69

# Blind-spot offset added to every depth, in cm. 0 = no correction (default).
# See the root README, "Instrument corrections", before changing this.
BLIND_SPOT_OFFSET_CM = 0.0

In [ ]:
scans = cb.load_settling_column_scans(SCANS_DIR, blind_spot_offset_cm=BLIND_SPOT_OFFSET_CM)
if not scans:
    raise SystemExit(f"No scan folders found in {SCANS_DIR}. See README.md, 'Analyzing Data'.")

degrees = {deg for deg, _ in scans.values()}
if len(degrees) > 1:
    print(f"WARNING: scans use different scan angles {sorted(degrees)}; plot them separately.")
scan_degree = degrees.pop() if len(degrees) == 1 else None

heights = np.array(list(scans.keys()))
stats = {h: cb.depth_stats(scan) for h, (_, scan) in scans.items()}
mean_depth = np.array([stats[h].mean_cm for h in heights])
sd_depth = np.array([stats[h].sd_cm for h in heights])

print(f"Blind-spot offset applied: {BLIND_SPOT_OFFSET_CM} cm")
print(f"{'sediment (cm)':>13} {'mean (cm)':>10} {'SD (cm)':>8} {'n':>6} {'settled (cm)':>13}  processing")
for h, (_, scan) in scans.items():
    st = stats[h]
    settled, _n = cb.settled_depth(scan)
    print(f"{h:13.1f} {st.mean_cm:10.2f} {st.sd_cm:8.2f} {st.n:6d} {settled:13.2f}  {cb.processing_summary(scan)}")

### Uncertainty

Error bars are **±1 sample standard deviation** (`ddof=1`) of the individual depth readings in each scan. The table above gives the number of readings, *n*. This describes the scatter of single readings, not the uncertainty of the mean; the standard error of the mean is SD/√n. Readings within a scan are autocorrelated (about 14 per second), so SD/√n understates that uncertainty.

In [ ]:
plt.figure()
plt.plot(heights, mean_depth, marker="x", linestyle="-", color="green", label="Sonar mean depth (uncorrected)" if BLIND_SPOT_OFFSET_CM == 0 else f"Sonar mean depth (+{BLIND_SPOT_OFFSET_CM} cm offset)")
plt.errorbar(heights, mean_depth, yerr=sd_depth, fmt="none", ecolor="orange", capsize=5, label="±1 SD (sample)")
plt.plot(heights, OBSERVED_WATER_HEIGHT_CM - heights, "r--", label="Expected depth")

ax = plt.gca()
ax.yaxis.set_minor_locator(MultipleLocator(1))
ax.tick_params(axis="y", which="minor", length=4, color="black", direction="in")
ax.set_ylim(35, 75)

angle = f" ({scan_degree:g}° scan angle)" if scan_degree is not None else ""
plt.xlabel("Sediment height (cm)")
plt.ylabel("Depth below transducer (cm)")
plt.title(f"Average bathymetry depth vs. expected depth{angle}")
plt.grid(True)
plt.xticks(heights)
plt.legend(framealpha=1.0)
plt.savefig("average_bathymetry.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def format_time(seconds, pos):
    return f"{int(seconds // 60):02d}:{int(seconds % 60):02d}"


colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

plt.figure(figsize=(14, 6))
ax = plt.gca()
for index, (height, (_, scan)) in enumerate(scans.items()):
    color = colors[index % len(colors)]
    ax.plot(scan.elapsed_s, scan.depths_cm, color=color, label=f"{height:.1f} cm sediment (n={scan.n})")

    # Expected depth for this scan, marked just outside the right edge.
    expected = OBSERVED_WATER_HEIGHT_CM - height
    ax.annotate("", xy=(1.01, expected), xycoords=("axes fraction", "data"),
                xytext=(1.03, expected), textcoords=("axes fraction", "data"),
                arrowprops=dict(arrowstyle="-", color=color, linewidth=2))
    ax.annotate(f"{expected:.1f} cm", xy=(1.035, expected), xycoords=("axes fraction", "data"),
                va="center", ha="left", color=color, fontsize=10)

ax.set_xlabel("Scan duration (mm:ss)")
ax.set_ylabel("Depth below transducer (cm)")
ax.set_title("CHIRP bathymetry depth over time")
ax.legend(frameon=True, framealpha=1.0, loc="upper right")
ax.xaxis.set_major_locator(MultipleLocator(30))
ax.xaxis.set_major_formatter(FuncFormatter(format_time))
ax.yaxis.set_minor_locator(MultipleLocator(2))
ax.tick_params(axis="y", which="minor", length=4, color="black", direction="in")
ax.grid(True, which="major", linestyle="--", linewidth=1, color="black", alpha=1)
ax.margins(x=0)
ax.set_ylim(15, 75)
plt.xticks(rotation=45)
ax.invert_yaxis()
plt.savefig("sonar_data.png", dpi=150, bbox_inches="tight")
plt.show()

## Manual plot (any files)

Plot any `bathymetry.csv` files by path, without the folder naming convention. Horizontal lines can mark reference depths, for example a depth measured by hand.

In [ ]:
def plot_bathymetry_manual(files_and_labels, title, hlines=None, ylim=None,
                           output_file="manual_bathymetry.png",
                           blind_spot_offset_cm=BLIND_SPOT_OFFSET_CM):
    plt.figure(figsize=(14, 6))
    ax = plt.gca()
    for file_path, label in files_and_labels:
        scan = cb.read_bathymetry(file_path, blind_spot_offset_cm)
        if scan.n == 0:
            print(f"{file_path}: no usable rows")
            continue
        print(f"{label}: {cb.processing_summary(scan)}")
        ax.plot(scan.elapsed_s, scan.depths_cm, label=f"{label} (n={scan.n})")

    for y_value, hline_label, color in hlines or []:
        ax.axhline(y=y_value, linestyle=":", linewidth=1.5, color=color, label=hline_label)

    ax.set_xlabel("Scan duration (mm:ss)")
    ax.set_ylabel("Depth below transducer (cm)")
    ax.set_title(title)
    ax.legend(frameon=True, framealpha=1.0, loc="upper right")
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.xaxis.set_major_formatter(FuncFormatter(format_time))
    ax.yaxis.set_minor_locator(MultipleLocator(2))
    ax.tick_params(axis="y", which="minor", length=4, color="black", direction="in")
    ax.grid(True, which="major", linestyle="--", linewidth=1, color="black", alpha=1)
    ax.margins(x=0)
    if ylim:
        ax.set_ylim(*ylim)
    plt.xticks(rotation=45)
    ax.invert_yaxis()
    plt.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.show()


# Replace with your own file paths, labels, title and reference depths.
example = SCANS_DIR / "scan_data_dredging_site" / "bathymetry.csv"
if example.exists():
    plot_bathymetry_manual(
        files_and_labels=[(example, "CHIRP scan data")],
        title="Dredging site calibration bathymetry data",
        hlines=[(53, "Measured depth", "red")],
        ylim=(42, 60),
    )
else:
    print(f"Example file {example} not found; edit the paths in this cell.")